# Inference data preparation: transforming `inferencia_df`

This notebook is standalone (it does not depend on the training notebook). We apply to `inferencia_df` (sales of October 25th to November 30th, 2025) exactly the same steps that were applied to `df` in `entrenamiento.ipynb`, so it ends up with the same variables and structure and is ready for the forecasting model:

1. **Load and check** the inference data. It already includes the competitors' prices, so no merge is needed. `unidades_vendidas` and `ingresos` are only known for October (the November values are what we want to forecast).
2. **Calendar variables**, with the same logic as in training, using the Black Friday of 2025.
3. **Lags and moving average**, computed per product and year **before** removing October, because the first days of November depend on the sales of October 25th-31st. Rows with nulls are **not** dropped and nulls are **not** filled.
4. **Price variables** (`descuento_porcentaje`, `precio_competencia`, `ratio_precio`) with the same formulas as in training.
5. **One-hot encoding** with the saved `aplicar_onehot` function and `models/onehot_columnas.csv` (no new `get_dummies` fitting).
6. **Keep only November** (30 days x 24 products = 720 rows), check the structure against `df` and save the result.


In [1]:
# Data manipulation and numerical computing
import pandas as pd
import numpy as np

# Visualization (full matplotlib package, its pyplot interface and seaborn)
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

# Machine learning
import sklearn

# Web app framework
import streamlit as st

# Public holidays by country
import holidays

# Import Path to handle the folder and file paths, and joblib to load the saved model
from pathlib import Path
import joblib

In [2]:
# Define the paths (relative to the notebooks folder) of the saved artifacts and the processed data
ruta_modelos = Path("../models")
ruta_onehot = ruta_modelos / "onehot_columnas.csv"
ruta_modelo_final = ruta_modelos / "modelo_final.joblib"
ruta_procesado = Path("../data/processed")
ruta_procesado.mkdir(parents=True, exist_ok=True)

# Read the saved one-hot columns (single column named "columna", in the training order)
cols_onehot = pd.read_csv(ruta_onehot)["columna"].tolist()

# Define the raw columns, the competitors and the calendar columns used in the training notebook
cols_ventas = ["fecha", "producto_id", "nombre", "categoria", "subcategoria", "precio_base", "es_estrella",
               "unidades_vendidas", "precio_venta", "ingresos"]
cols_competidores = ["Amazon", "Decathlon", "Deporvillage"]
cols_calendario = ["fecha", "año", "mes", "dia_mes", "dia_semana", "dia_año", "semana_año", "trimestre", "es_fin_semana",
                   "es_festivo", "es_vispera_festivo", "es_black_friday", "es_cyber_monday", "es_semana_black_friday",
                   "dias_hasta_black_friday", "es_inicio_mes", "es_fin_mes"]

# Define the lag and moving average columns created from the units sold
cols_nuevas = [f"unidades_lag_{lag}" for lag in range(1, 8)] + ["unidades_media_movil_7"]

# Define the target of the model
target = "unidades_vendidas"

In [3]:
def aplicar_onehot(df_nuevo):
    """Apply the saved one-hot encoding to new data, returning exactly the saved columns in the saved order."""
    # Build an auxiliary dataframe with only the descriptive variables, renamed with the _h suffix
    auxiliar = df_nuevo[["nombre", "categoria", "subcategoria"]].rename(
        columns={"nombre": "nombre_h", "categoria": "categoria_h", "subcategoria": "subcategoria_h"}
    )

    # Apply one-hot encoding only to the auxiliary dataframe (df_nuevo is not modified)
    codificado = pd.get_dummies(auxiliar, dtype=int)

    # Read the saved columns and align: new categories are ignored and missing ones are filled with zeros
    columnas_guardadas = pd.read_csv(ruta_onehot)["columna"].tolist()
    return codificado.reindex(columns=columnas_guardadas, fill_value=0)

In [4]:
# Load the 2025 inference sales data (path relative to the notebooks folder)
inferencia_df = pd.read_csv("../data/raw/inferencia/ventas_2025_inferencia.csv")

# Convert the date column to datetime, as it was done for the training dataframes
inferencia_df["fecha"] = pd.to_datetime(inferencia_df["fecha"])

# Check the shape, the date range and that (fecha, producto_id) is a unique key
print(inferencia_df.shape)
print("Range:", inferencia_df["fecha"].min().date(), "->", inferencia_df["fecha"].max().date())
print("Key duplicates:", inferencia_df.duplicated(subset=["fecha", "producto_id"]).sum())

# Check that the raw columns are the sales columns plus the competitors' prices
print("Raw columns as expected:", inferencia_df.columns.tolist() == cols_ventas + cols_competidores)

# Count the non-null values per month: the sales are only known in October
print(inferencia_df.groupby(inferencia_df["fecha"].dt.month)[["unidades_vendidas", "ingresos", "precio_venta", "Amazon"]].count())

(888, 13)
Range: 2025-10-25 -> 2025-11-30
Key duplicates: 0
Raw columns as expected: True
       unidades_vendidas  ingresos  precio_venta  Amazon
fecha                                                   
10                   168       168           168     168
11                     0         0           720     720


In [5]:
# Extract the basic date parts
inferencia_df["año"] = inferencia_df["fecha"].dt.year
inferencia_df["mes"] = inferencia_df["fecha"].dt.month
inferencia_df["dia_mes"] = inferencia_df["fecha"].dt.day
inferencia_df["dia_semana"] = inferencia_df["fecha"].dt.dayofweek  # Monday=0 ... Sunday=6
inferencia_df["dia_año"] = inferencia_df["fecha"].dt.dayofyear
inferencia_df["semana_año"] = inferencia_df["fecha"].dt.isocalendar().week.astype(int)
inferencia_df["trimestre"] = inferencia_df["fecha"].dt.quarter

# Flag the weekends (Saturday and Sunday)
inferencia_df["es_fin_semana"] = (inferencia_df["dia_semana"] >= 5).astype(int)

# Build the set of Spanish national holidays for the years in the inference data
festivos_es_inf = set(holidays.Spain(years=inferencia_df["año"].unique().tolist()).keys())

# Flag the holidays and the eve of a holiday
inferencia_df["es_festivo"] = inferencia_df["fecha"].dt.date.isin(festivos_es_inf).astype(int)
inferencia_df["es_vispera_festivo"] = (inferencia_df["fecha"] + pd.Timedelta(days=1)).dt.date.isin(festivos_es_inf).astype(int)

# Compute the Black Friday of 2025 with the same rule as in training (day after the 4th Thursday of November)
black_fridays_inf = pd.date_range("2025-01-01", "2025-12-31", freq="WOM-4THU") + pd.Timedelta(days=1)
black_fridays_inf = black_fridays_inf[black_fridays_inf.month == 11]
print("Black Friday 2025:", black_fridays_inf[0].date(), "|", black_fridays_inf[0].day_name())

# Map it to every row
black_friday_por_año_inf = pd.Series(black_fridays_inf, index=black_fridays_inf.year)
fecha_black_friday_inf = inferencia_df["año"].map(black_friday_por_año_inf)

# Flag Black Friday and Cyber Monday (the Monday right after Black Friday)
inferencia_df["es_black_friday"] = (inferencia_df["fecha"] == fecha_black_friday_inf).astype(int)
inferencia_df["es_cyber_monday"] = (inferencia_df["fecha"] == fecha_black_friday_inf + pd.Timedelta(days=3)).astype(int)

# Flag the Black Friday week (Monday to Sunday) and compute the signed days to Black Friday
inferencia_df["es_semana_black_friday"] = (inferencia_df["semana_año"] == fecha_black_friday_inf.dt.isocalendar().week.astype(int)).astype(int)
inferencia_df["dias_hasta_black_friday"] = (fecha_black_friday_inf - inferencia_df["fecha"]).dt.days

# Flag the first and last 3 days of the month
inferencia_df["es_inicio_mes"] = (inferencia_df["dia_mes"] <= 3).astype(int)
inferencia_df["es_fin_mes"] = ((inferencia_df["fecha"].dt.days_in_month - inferencia_df["dia_mes"]) < 3).astype(int)

# Check the flagged dates
display(inferencia_df[cols_calendario].drop_duplicates("fecha").query("es_festivo == 1 or es_black_friday == 1 or es_cyber_monday == 1"))

Black Friday 2025: 2025-11-28 | Friday


,fecha,año,mes,dia_mes,dia_semana,dia_año,semana_año,trimestre,es_fin_semana,es_festivo,es_vispera_festivo,es_black_friday,es_cyber_monday,es_semana_black_friday,dias_hasta_black_friday,es_inicio_mes,es_fin_mes
168,2025-11-01,2025,11,1,5,305,44,4,1,1,0,0,0,0,27,1,0
816,2025-11-28,2025,11,28,4,332,48,4,0,0,0,1,0,1,0,0,1


In [6]:
# Sort by product and date so the shifts follow the time order inside each product
inferencia_df = inferencia_df.sort_values(["producto_id", "fecha"]).reset_index(drop=True)

# Group by product and year, as in training (October is still in the dataframe, so the early November lags can be computed)
grupos_inf = inferencia_df.groupby(["producto_id", "año"])["unidades_vendidas"]

# Create the lags 1 to 7 of the units sold
for lag in range(1, 8):
    inferencia_df[f"unidades_lag_{lag}"] = grupos_inf.shift(lag)

# Create the 7-day moving average of the previous days (shifted by 1 to exclude the current day)
inferencia_df["unidades_media_movil_7"] = grupos_inf.transform(lambda s: s.shift(1).rolling(window=7).mean())

# No dropna and no filling: count the nulls left in the new variables in November
nov_inf = inferencia_df["mes"] == 11
print("November rows:", nov_inf.sum())
print(inferencia_df.loc[nov_inf, cols_nuevas].isna().sum())

# Check that the lags of the first days of November come from October 25th-31st (example with PROD_001)
inferencia_df[(inferencia_df["producto_id"] == "PROD_001") & (inferencia_df["fecha"].between("2025-10-30", "2025-11-03"))][["fecha", "unidades_vendidas", "unidades_lag_1", "unidades_lag_2", "unidades_media_movil_7"]]

November rows: 720
unidades_lag_1            696
unidades_lag_2            672
unidades_lag_3            648
unidades_lag_4            624
unidades_lag_5            600
unidades_lag_6            576
unidades_lag_7            552
unidades_media_movil_7    696
dtype: int64


,fecha,unidades_vendidas,unidades_lag_1,unidades_lag_2,unidades_media_movil_7
5,2025-10-30,10.0,14.0,15.0,NaN
6,2025-10-31,14.0,10.0,14.0,NaN
7,2025-11-01,NaN,14.0,10.0,16.428571
8,2025-11-02,NaN,NaN,14.0,NaN
9,2025-11-03,NaN,NaN,NaN,NaN


In [7]:
# Create the discount percentage with the same formula and sign as in training
inferencia_df["descuento_porcentaje"] = (inferencia_df["precio_venta"] - inferencia_df["precio_base"]) / inferencia_df["precio_base"] * 100

# Create the average price of the competitors and the price ratio
inferencia_df["precio_competencia"] = inferencia_df[cols_competidores].mean(axis=1)
inferencia_df["ratio_precio"] = inferencia_df["precio_venta"] / inferencia_df["precio_competencia"]

# Drop the individual competitors' price columns
inferencia_df = inferencia_df.drop(columns=cols_competidores)

# Check the new variables
print(inferencia_df[["descuento_porcentaje", "precio_competencia", "ratio_precio"]].describe())

       descuento_porcentaje  precio_competencia  ratio_precio
count            888.000000          888.000000    888.000000
mean              -0.419538          116.271062      1.043379
std                2.761753          155.138858      0.046390
min              -15.000000           18.473333      0.871923
25%               -0.877692           46.133333      1.016850
50%                0.000000           70.285000      1.040254
75%                0.762857          111.060833      1.065539
max               10.332500          837.193333      1.203810


In [8]:
# Apply the saved one-hot encoding (44 columns, same order as in training) and append it to the dataframe
inferencia_df = pd.concat([inferencia_df, aplicar_onehot(inferencia_df)], axis=1)

# Check the one-hot columns are exactly the training ones
print(len(cols_onehot), "one-hot columns expected | same columns and order:", inferencia_df.columns[-len(cols_onehot):].tolist() == cols_onehot)

44 one-hot columns expected | same columns and order: True


In [9]:
# Load the training dataframe saved by the training notebook, used only as the structure reference
df_ref = pd.read_csv(ruta_procesado / "df.csv", parse_dates=["fecha"])

# Rebuild the predictors as in training (all columns except fecha, ingresos, the target and the text columns)
cols_texto = df_ref.select_dtypes(exclude=["number", "bool", "datetime"]).columns.tolist()
predictoras = [col for col in df_ref.columns if col not in ["fecha", "ingresos", target] + cols_texto]

# Check that they are the same predictors, in the same order, that the saved final model was trained with
modelo_final = joblib.load(ruta_modelo_final)
print(len(predictoras), "predictors | same as the saved model:", predictoras == list(modelo_final.feature_names_in_))

74 predictors | same as the saved model: True


In [10]:
# Keep only November, after all the transformations are done, and restore the original order (date and product)
inferencia_df = inferencia_df[inferencia_df["mes"] == 11].sort_values(["fecha", "producto_id"]).reset_index(drop=True)
print("Rows:", len(inferencia_df), "| Expected 720:", len(inferencia_df) == 720)
print("Dates:", inferencia_df["fecha"].min().date(), "->", inferencia_df["fecha"].max().date(), "| Products:", inferencia_df["producto_id"].nunique())

# Check 1: same columns, in the same order, as df (the training dataframe)
print("Same columns and order as df:", inferencia_df.columns.tolist() == df_ref.columns.tolist())

# Check 2: all the predictors are present, in the same order as in training
print("All predictors present:", set(predictoras).issubset(inferencia_df.columns))
print("Same predictors order:", [col for col in inferencia_df.columns if col in predictoras] == predictoras)
print("Predictors castable to float, as the model input in training:", inferencia_df[predictoras].astype(float).shape == (len(inferencia_df), len(predictoras)))

# Check 3: the target and ingresos are not predictors, and they are empty in November (they are what we want to forecast)
print("Target / ingresos in predictors:", [col for col in [target, "ingresos"] if col in predictoras])
print("Empty unidades_vendidas and ingresos:", inferencia_df[[target, "ingresos"]].isna().all().all())

# Check 4: the only predictors built from unidades_vendidas are the lags and the moving average (known up to October 31st)
print("Predictors built from the units sold:", [col for col in predictoras if col.startswith("unidades_")])

Rows: 720 | Expected 720: True
Dates: 2025-11-01 -> 2025-11-30 | Products: 24
Same columns and order as df: True
All predictors present: True
Same predictors order: True
Predictors castable to float, as the model input in training: True
Target / ingresos in predictors: []
Empty unidades_vendidas and ingresos: True
Predictors built from the units sold: ['unidades_lag_1', 'unidades_lag_2', 'unidades_lag_3', 'unidades_lag_4', 'unidades_lag_5', 'unidades_lag_6', 'unidades_lag_7', 'unidades_media_movil_7']


In [11]:
# Save the transformed inference dataframe as a CSV without the index
ruta_inferencia = ruta_procesado / "inferencia_df_transformado.csv"
inferencia_df.to_csv(ruta_inferencia, index=False)

# Read the saved file back, parsing the date column
inferencia_leido = pd.read_csv(ruta_inferencia, parse_dates=["fecha"])

# Check the shapes and that the columns are the same and in the same order
print("Saved file shape:", inferencia_leido.shape)
print("inferencia_df shape:", inferencia_df.shape)
print("Same columns and order:", inferencia_leido.columns.tolist() == inferencia_df.columns.tolist())
print("Same columns and order as df:", inferencia_leido.columns.tolist() == df_ref.columns.tolist())

Saved file shape: (720, 81)
inferencia_df shape: (720, 81)
Same columns and order: True
Same columns and order as df: True
